In [23]:
import pandas as pd
import numpy as np

# Preprocesamiento y Pipelines
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas y guardado
from sklearn.metrics import accuracy_score
import pickle

In [24]:
# Cargar el dataset sin las transformaciones manuales previas
df = pd.read_csv('./data/titanic_clean.csv')

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.compose import ColumnTransformer

# 1. Pipeline para variables categóricas (Sex, Embarked):
# Imputa moda + OneHotEncoder
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 2. Pipeline para variables numéricas especiales (Age, Fare):
# Imputa mediana + QuantileTransformer
num_quantile_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('quantile', QuantileTransformer(output_distribution='normal', n_quantiles=500))
])

# 3. Pipeline para el resto de variables numéricas (Pclass, SibSp, Parch):
# Rellena cualquier nulo con la mediana sin transformaciones extra
num_passthrough_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# ColumnTransformer integrando todo (¡Ya no usamos passthrough directo!)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, ['Sex', 'Embarked']),
        ('quantile_num', num_quantile_pipeline, ['Age', 'Fare']),
        ('passthrough_num', num_passthrough_pipeline, ['Pclass', 'SibSp', 'Parch'])
    ]
)

In [26]:
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'model__C': [0.01, 0.1, 1, 10, 100],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear', 'saga'],
            'model__max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    }
}

In [27]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

In [28]:
for nombre, info_modelo in modelos.items():
    # Pipeline limpio: preprocessor -> scaler -> modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', MinMaxScaler()),
        ('model', info_modelo['modelo'])
    ])
    
    # Inicializar GridSearchCV
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Evaluar
    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)
    
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })
    
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Imprimir resultados y guardar el archivo .pkl
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

print('-----------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

with open('pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)


c:\Users\Alicia Vaca\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
6      Clasificador K-Nearest Neighbors       0.84
4     Clasificador de Gradient Boosting       0.82
8                     Clasificador LGBM       0.82
0                   Regresión Logística       0.81
1   Clasificador de Vectores de Soporte       0.81
5                 Clasificador AdaBoost       0.81
7                  Clasificador XGBoost       0.81
2     Clasificador de Árbol de Decisión       0.80
3    Clasificador de Bosques Aleatorios       0.80
9                            GaussianNB       0.79
10             Clasificador Naive Bayes       0.78
-----------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.84
